# LUCJ circuit generation and transpilation (solvent)

For every geometry shift this notebook:
1. loads the UCCSD amplitudes from `../../classical_methods/solvent/charge<C>_spin<S>/orbitals/shift_<shift>/uccsd_amps_<as_label>.npz`,
2. builds a spin-unbalanced LUCJ ansatz (`ffsim.UCJOpSpinUnbalanced.from_t_amplitudes`, `n_reps=1`, zig-zag interaction pairs) on top of the Hartree-Fock state,
3. saves the logical circuit to `charge<C>_spin<S>/lucj_circuits/qubits_<N>_shift_<shift>.qpy`,
4. transpiles it for the target IBM backend (`optimization_level=3`, zig-zag initial layout on the heavy-hex lattice chosen by a lightweight error score, `ffsim.qiskit.PRE_INIT` passes in the `pre_init` stage) and saves it to `charge<C>_spin<S>/transpiler_pass_initial/<N>_qubits/qubits_<N>_shift_<shift>_<backend>.qpy`.

Requires a saved IBM Quantum account (`QiskitRuntimeService.save_account(...)`).

**Note:** re-running overwrites the shipped circuits. The initial layout is scored using the backend calibration data at run time, so regenerated transpiled circuits will in general differ from the shipped ones.

In [ ]:
import os
import sys

import json
import pyscf
from pyscf import lib
from pyscf.scf import chkfile
from pyscf import ao2mo, tools 
import numpy as np
from pathlib import Path
import ffsim
import time
from qiskit import QuantumCircuit, QuantumRegister
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_aer import AerSimulator
from qiskit_addon_sqd.counts import counts_to_arrays

from matplotlib import pyplot as plt

from copy import deepcopy
import pickle
from qiskit_ibm_runtime import RuntimeEncoder,RuntimeDecoder
import qiskit.primitives.containers.bit_array as bit_array

from scipy import linalg as LA

from pyscf.mcscf import avas

from pyscf import gto, scf, mp, mcscf, cc, ci, tools, ao2mo

import scipy

from qiskit import qpy

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

In [ ]:
from typing import Sequence
 
import rustworkx
from qiskit.providers import BackendV2
from rustworkx import NoEdgeBetweenNodes, PyGraph
 
IBM_TWO_Q_GATES = {"cx", "ecr", "cz"}
 
 
def create_linear_chains(num_orbitals: int) -> PyGraph:
    """In zig-zag layout, there are two linear chains (with connecting qubits between
    the chains). This function creates those two linear chains: a rustworkx PyGraph
    with two disconnected linear chains. Each chain contains `num_orbitals` number
    of nodes, that is, in the final graph there are `2 * num_orbitals` number of nodes.
 
    Args:
        num_orbitals (int): Number orbitals or nodes in each linear chain. They are
            also known as alpha-alpha interaction qubits.
 
    Returns:
        A rustworkx.PyGraph with two disconnected linear chains each with `num_orbitals`
            number of nodes.
    """
    G = rustworkx.PyGraph()
 
    for n in range(num_orbitals):
        G.add_node(n)
 
    for n in range(num_orbitals - 1):
        G.add_edge(n, n + 1, None)
 
    for n in range(num_orbitals, 2 * num_orbitals):
        G.add_node(n)
 
    for n in range(num_orbitals, 2 * num_orbitals - 1):
        G.add_edge(n, n + 1, None)
 
    return G
 
 
def create_lucj_zigzag_layout(
    num_orbitals: int, backend_coupling_graph: PyGraph
) -> tuple[PyGraph, int]:
    """This function creates the complete zigzag graph that 'can be mapped' to an IBM QPU with
    heavy-hex connectivity (the zigzag must be an isomorphic sub-graph to the QPU/backend
    coupling graph for it to be mapped).
    The zigzag pattern includes both linear chains (alpha-alpha interactions) and connecting
    qubits between the linear chains (alpha-beta interactions).
 
    Args:
        num_orbitals (int): Number of orbitals, that is, number of nodes in each alpha-alpha linear chain.
        backend_coupling_graph (PyGraph): The coupling graph of the backend on which the LUCJ ansatz
            will be mapped and run. This function takes the coupling graph as a undirected
            `rustworkx.PyGraph` where there is only one 'undirected' edge between two nodes,
            that is, qubits. Usually, the coupling graph of a IBM backend is directed (for example, Eagle devices
            such as ibm_brisbane) or may have two edges between two nodes (for example, Heron `ibm_torino`).
            A user needs to be make such graphs undirected and/or remove duplicate edges to make them
            compatible with this function.
 
    Returns:
        G_new (PyGraph): The graph with IBM backend compliant zigzag pattern.
        num_alpha_beta_qubits (int): Number of connecting qubits between the linear chains
            in the zigzag pattern. While we want as many connecting (alpha-beta) qubits between
            the linear (alpha-alpha) chains, we cannot accommodate all due to qubit and connectivity
            constraints of backends. This is the maximum number of connecting qubits the zigzag pattern
            can have while being backend compliant (that is, isomorphic to backend coupling graph).
    """
    isomorphic = False
    G = create_linear_chains(num_orbitals=num_orbitals)
 
    num_iters = num_orbitals
    while not isomorphic:
        G_new = G.copy()
        num_alpha_beta_qubits = 0
        for n in range(num_iters):
            if n % 4 == 0:
                new_node = 2 * num_orbitals + num_alpha_beta_qubits
                G_new.add_node(new_node)
                G_new.add_edge(n, new_node, None)
                G_new.add_edge(new_node, n + num_orbitals, None)
                num_alpha_beta_qubits = num_alpha_beta_qubits + 1
        isomorphic = rustworkx.is_subgraph_isomorphic(
            backend_coupling_graph, G_new
        )
        num_iters -= 1
 
    return G_new, num_alpha_beta_qubits
 
 
def lightweight_layout_error_scoring(
    backend: BackendV2,
    virtual_edges: Sequence[Sequence[int]],
    physical_layouts: Sequence[int],
    two_q_gate_name: str,
) -> list[list[list[int], float]]:
    """Lightweight and heuristic function to score isomorphic layouts. There can be many zigzag patterns,
    each with different set of physical qubits, that can be mapped to a backend. Some of them may
    include less noise qubits and couplings than others. This function computes a simple error score
    for each such layout. It sums up 2Q gate error for all couplings in the zigzag pattern (layout) and
    measurement of errors of physical qubits in the layout to compute the error score.
 
    Note:
        This lightweight scoring can be refined using concepts such as mapomatic.
 
    Args:
        backend (BackendV2): A backend.
        virtual_edges (Sequence[Sequence[int]]): Edges in the device compliant zigzag pattern where
            nodes are numbered from 0 to (2 * num_orbitals + num_alpha_beta_qubits).
        physical_layouts (Sequence[int]): All physical layouts of the zigzag pattern that are isomorphic
            to each other and to the larger backend coupling map.
        two_q_gate_name (str): The name of the two-qubit gate of the backend. The name is used for fetching
            two-qubit gate error from backend properties.
 
    Returns:
        scores (list): A list of lists where each sublist contains two items. First item is the layout, and
            second item is a float representing error score of the layout. The layouts in the `scores` are
            sorted in the ascending order of error score.
    """
    props = backend.properties()
    scores = []
    for layout in physical_layouts:
        total_2q_error = 0
        for edge in virtual_edges:
            physical_edge = (layout[edge[0]], layout[edge[1]])
            try:
                ge = props.gate_error(two_q_gate_name, physical_edge)
            except Exception:
                ge = props.gate_error(two_q_gate_name, physical_edge[::-1])
            total_2q_error += ge
        total_measurement_error = 0
        for qubit in layout:
            meas_error = props.readout_error(qubit)
            total_measurement_error += meas_error
        scores.append([layout, total_2q_error + total_measurement_error])
 
    return sorted(scores, key=lambda x: x[1])
 
 
def _make_backend_cmap_pygraph(backend: BackendV2) -> PyGraph:
    graph = backend.coupling_map.graph
    if not graph.is_symmetric():
        graph.make_symmetric()
    backend_coupling_graph = graph.to_undirected()
 
    edge_list = backend_coupling_graph.edge_list()
    removed_edge = []
    for edge in edge_list:
        if set(edge) in removed_edge:
            continue
        try:
            backend_coupling_graph.remove_edge(edge[0], edge[1])
            removed_edge.append(set(edge))
        except NoEdgeBetweenNodes:
            pass
 
    return backend_coupling_graph
 
 
def get_zigzag_physical_layout(
    num_orbitals: int, backend: BackendV2, score_layouts: bool = True
) -> tuple[list[int], int]:
    """The main function that generates the zigzag pattern with physical qubits that can be used
    as an `intial_layout` in a preset passmanager/transpiler.
 
    Args:
        num_orbitals (int): Number of orbitals.
        backend (BackendV2): A backend.
        score_layouts (bool): Optional. If `True`, it uses the `lightweight_layout_error_scoring`
            function to score the isomorphic layouts and returns the layout with less erroneous qubits.
            If `False`, returns the first isomorphic subgraph.
 
    Returns:
        A tuple of device compliant layout (list[int]) with zigzag pattern and an int representing
            number of alpha-beta-interactions.
    """
    backend_coupling_graph = _make_backend_cmap_pygraph(backend=backend)
 
    G, num_alpha_beta_qubits = create_lucj_zigzag_layout(
        num_orbitals=num_orbitals,
        backend_coupling_graph=backend_coupling_graph,
    )
 
    isomorphic_mappings = rustworkx.vf2_mapping(
        backend_coupling_graph, G, subgraph=True
    )
    isomorphic_mappings = list(isomorphic_mappings)
 
    edges = list(G.edge_list())
 
    layouts = []
    for mapping in isomorphic_mappings:
        initial_layout = [None] * (2 * num_orbitals + num_alpha_beta_qubits)
        for key, value in mapping.items():
            initial_layout[value] = key
        layouts.append(initial_layout)
 
    two_q_gate_name = IBM_TWO_Q_GATES.intersection(
        backend.configuration().basis_gates
    ).pop()
 
    if score_layouts:
        scores = lightweight_layout_error_scoring(
            backend=backend,
            virtual_edges=edges,
            physical_layouts=layouts,
            two_q_gate_name=two_q_gate_name,
        )
 
        return scores[0][0][:-num_alpha_beta_qubits], num_alpha_beta_qubits
 
    return layouts[0][:-num_alpha_beta_qubits], num_alpha_beta_qubits

In [ ]:
def save_lucj_circuit(qc,shift,DATA_DIR):
    folder=Path(DATA_DIR+f'/lucj_circuits')
    folder.mkdir(parents=True,exist_ok=True)
    fname=str(folder)+f'/qubits_{qc.num_qubits}_shift_{shift}'
    with open(f"{fname}.qpy", "wb") as file:
        qpy.dump(qc, file)
    return

def save_transpiled_circuit(qc,backend,num_qubits,shift,DATA_DIR):
    folder=Path(DATA_DIR+f'/transpiler_pass_initial/{num_qubits}_qubits')
    folder.mkdir(parents=True,exist_ok=True)
    fname=str(folder)+f'/qubits_{num_qubits}_shift_{shift}_{backend}'
    with open(f"{fname}.qpy", "wb") as file:
        qpy.dump(qc, file)
    return

### MAIN

In [ ]:
service=QiskitRuntimeService()
service.backends()
backend_name="ibm_pittsburgh"
backend = service.backend(backend_name,use_fractional_gates=False)

### Parameters
Edit the cell below for the state / active space to generate:

| State | `charge` | `spin_number` (2S) | `num_orbitals` | `(num_elec_a, num_elec_b)` | qubits |
|---|---|---|---|---|---|
| Co(III) singlet | 3 | 0 | 17 / 19 / 23 / 25 | (11,11) / (12,12) / (14,14) / (15,15) | 34 / 38 / 46 / 50 |
| Co(III) quintet | 3 | 4 | 17 / 19 / 23 / 25 | (13,9) / (14,10) / (16,12) / (17,13) | 34 / 38 / 46 / 50 |
| Co(II) doublet | 2 | 1 | 17 | (12,11) | 34 |
| Co(II) quartet | 2 | 3 | 17 | (13,10) | 34 |

In [ ]:
shift_array=['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']
charge=3
spin_number=4
medium='solvent'
num_orbitals=17
num_elec_a,num_elec_b=13,9
DATA_DIR=f'charge{charge}_spin{spin_number}'
as_label = 'norb_%d_nelec_%d_%d'%(num_orbitals,num_elec_a,num_elec_b)
use_threshold=False

In [ ]:
for (j,shift) in enumerate(shift_array):
    print("SHIFT ",shift)
    ### Creating bare circuits
    data = np.load(f'../../classical_methods/{medium}/charge{charge}_spin{spin_number}/orbitals/shift_{shift}/uccsd_amps_{as_label}.npz')
    t1 = (data['t1a'], data['t1b'])
    t2= (data['t2aa'], data['t2ab'], data['t2bb'])

    if use_threshold==True:
        threshold=0.1
        t1=np.where(t1[0] > np.max(t1[0])*threshold, t1[0], 0),np.where(t1[1] > np.max(t1[1])*threshold, t1[1], 0)
        t2=np.where(t2[0] > np.max(t2[0])*threshold, t2[0], 0),np.where(t2[1] > np.max(t2[1])*threshold, t2[1], 0),np.where(t2[2] > np.max(t2[2])*threshold, t2[2], 0)
        factor=100
        t1=t1[0]*factor,t1[1]*factor
        t2=t2[0]*factor,t2[1]*factor,t2[2]*factor

    alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
    alpha_beta_indices = [(p, p) for p in range(0, num_orbitals, 1)] # every 4th opposite-spin-flavour connectivity, 2D heavy-hex QPU
    beta_beta_indices = [(p, p + 1) for p in range(num_orbitals - 1)]


    ucj_op = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
        t1=t1,
        t2=t2,
        n_reps=1,
        interaction_pairs=(alpha_alpha_indices, alpha_beta_indices, beta_beta_indices),
    )

    print("-"*10)

    print('Building circuit...')
    # create an empty quantum circuit
    qubits = QuantumRegister(num_orbitals*2, name="q")
    qc = QuantumCircuit(qubits)

    # prepare Hartree-Fock state as the reference state and append it to the quantum circuit
    qc.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, (num_elec_a, num_elec_b)), qubits)

    # apply the UCJ operator to the reference state
    qc.append(ffsim.qiskit.UCJOpSpinUnbalancedJW(ucj_op), qubits)

    qc.measure_all()

    save_lucj_circuit(qc,shift,DATA_DIR)

    print("-"*10)

    print('Transpiling circuit...')
    initial_layout, _ = get_zigzag_physical_layout(qc.num_qubits//2, backend=backend)

    pass_manager = generate_preset_pass_manager(
        optimization_level=3, backend=backend, initial_layout=initial_layout
    )

    # without PRE_INIT passes
    isa_circuit = pass_manager.run(qc)
    print(f"Gate counts (w/o pre-init passes): {isa_circuit.count_ops()}")

    # with PRE_INIT passes
    # We will use the circuit generated by this pass manager for hardware execution
    pass_manager.pre_init = ffsim.qiskit.PRE_INIT
    isa_circuit = pass_manager.run(qc)
    print(f"Gate counts (w/ pre-init passes): {isa_circuit.count_ops()}")

    cdepth1=isa_circuit.depth()
    print(f'Circuit depth after transpilation is {cdepth1}.')

    save_transpiled_circuit(isa_circuit,backend_name,qc.num_qubits,shift,DATA_DIR)
    
    print("-"*53)
